In [1]:
%matplotlib inline
import os
import sys
from pathlib import Path
from urllib.parse import quote_plus

import numpy as np
import pandas as pd

_PATH = Path.cwd()
_PROJECT_ROOT = _PATH
for _ in range(5):
    if (_PROJECT_ROOT / ".env").exists():
        break
    _PROJECT_ROOT = _PROJECT_ROOT.parent

_TRAINING_DIR = _PROJECT_ROOT / "Models" / "Training"
sys.path.insert(0, str(_TRAINING_DIR))
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv(_PROJECT_ROOT / ".env")

def get_engine():
    host = os.getenv("PGHOST", "localhost")
    port = os.getenv("PGPORT", "5432")
    user = os.getenv("PGUSER", "postgres")
    password = os.getenv("PGPASSWORD", "")
    dbname = os.getenv("PGDATABASE", "baseball")
    pw = quote_plus(password) if password else ""
    return create_engine(f"postgresql://{user}:{pw}@{host}:{port}/{dbname}")

from data_prep_batters import EVENTS_TO_IN_PLAY_RESULT
engine = get_engine()
print("Setup OK. EVENTS_TO_IN_PLAY_RESULT loaded.")

Setup OK. EVENTS_TO_IN_PLAY_RESULT loaded.


In [2]:
# Load batted balls; map events -> 4-class in-play (Out, Single, XBH, HR). Include stand + career for ISO/zone.
q = """
SELECT release_speed, plate_x, plate_z, launch_speed, launch_angle, events, stand, career_BA, career_SLG,
       game_date, game_year, home_team, release_spin_rate
FROM clean_statcast_with_batter
WHERE launch_speed IS NOT NULL AND launch_angle IS NOT NULL
"""
df = pd.read_sql(q, engine)
df["events_clean"] = df["events"].astype(str).str.strip().str.lower()
df["in_play_raw"] = df["events_clean"].map(EVENTS_TO_IN_PLAY_RESULT)
# Merge 2B and 3B into one "extra_base_hit" (XBH) class to reduce noise
def to_four_class(raw):
    if raw == "in_play_2b" or raw == "in_play_3b":
        return "extra_base_hit"
    return raw
df["in_play_result"] = df["in_play_raw"].map(to_four_class)
df = df.dropna(subset=["in_play_result"]).copy()

# Physics-derived distance_ft (drag + Magnus + elevation; matches simulator when use_drag=True)
from physics_engine import compute_trajectory

def _physics_distance(row):
    ps = row.get("release_spin_rate")
    try:
        psf = float(ps) if ps is not None and pd.notna(ps) else None
    except (TypeError, ValueError):
        psf = None
    ht = row.get("home_team")
    hts = str(ht).strip().upper() if ht is not None and pd.notna(ht) and str(ht).strip() else None
    if hts in ("NAN", "NONE", ""):
        hts = None
    traj = compute_trajectory(
        float(row["launch_speed"]), float(row["launch_angle"]), 0.0,
        use_drag=True,
        pitch_spin_rpm=psf,
        home_team=hts,
    )
    return traj["distance_ft"]

df["distance_ft"] = df.apply(_physics_distance, axis=1)

# Isolated Power: career_ISO = SLG - BA (pure extra-base measure; use ISO not SLG in model)
# PostgreSQL often returns lowercase column names (career_ba, career_slg)
career_slg = df["career_slg"] if "career_slg" in df.columns else df["career_SLG"]
career_ba = df["career_ba"] if "career_ba" in df.columns else df["career_BA"]
df["career_iso"] = (career_slg.astype(float) - career_ba.astype(float)).clip(lower=0)
df.loc[career_ba.isna() | career_slg.isna(), "career_iso"] = np.nan
df["career_iso"] = df["career_iso"].fillna(0.12)  # league-typical default if missing

# Zone × handedness: stand (L/R) and interactions with plate location
df["stand_L"] = (df["stand"].astype(str).str.upper().str.strip() == "L").astype(int)
df["plate_x_stand"] = df["plate_x"].astype(float) * (2 * df["stand_L"] - 1)  # L=+1, R=-1
df["plate_z_stand"] = df["plate_z"].astype(float) * (2 * df["stand_L"] - 1)

FINAL_OUTCOME_NAMES = ["in_play_out", "in_play_1b", "extra_base_hit", "in_play_hr"]
df["outcome_idx"] = df["in_play_result"].map(lambda x: FINAL_OUTCOME_NAMES.index(x))
print(f"Batted balls: {len(df)}. Outcome counts (4-class):\n{df['in_play_result'].value_counts().sort_index()}")

Batted balls: 358876. Outcome counts (4-class):
in_play_result
extra_base_hit     25749
in_play_1b         77779
in_play_hr         16955
in_play_out       238393
Name: count, dtype: int64


In [3]:
# Features: launch + trajectory + career_ISO (not SLG) + zone × handedness
# Career ISO = SLG - BA (pure power); ensure we use ISO not career_SLG
feature_cols = [
    "launch_speed", "launch_angle", "distance_ft",
    "career_iso",
    "plate_x", "plate_z", "stand_L", "plate_x_stand", "plate_z_stand",
]
feature_cols = [c for c in feature_cols if c != "career_SLG"]
if "career_iso" not in feature_cols:
    feature_cols.append("career_iso")
X = df[feature_cols].astype(float).fillna(0)
y = df["outcome_idx"].astype(int)

from sklearn.metrics import classification_report, f1_score
from temporal_split import temporal_train_val_test

train_df, val_df, test_df, split_meta = temporal_train_val_test(df, train_frac=0.7, val_frac=0.15)
print("Temporal split:", split_meta)
X_train = X.loc[train_df.index]
X_val = X.loc[val_df.index]
X_test = X.loc[test_df.index]
y_train = y.loc[train_df.index]
y_val = y.loc[val_df.index]
y_test = y.loc[test_df.index]
# Class weights: Out +20% to reduce underprediction; HR at 30% to reduce overprediction
CLASS_WEIGHTS = {"in_play_out": 0.6, "in_play_1b": 1.5, "extra_base_hit": 2.5, "in_play_hr": 1.05}
weight_map = {i: CLASS_WEIGHTS[FINAL_OUTCOME_NAMES[i]] for i in range(len(FINAL_OUTCOME_NAMES))}
sample_weight_train = np.array([weight_map[yi] for yi in y_train])
print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

Temporal split: {'years_present': [2023, 2024, 2025], 'split_kind': 'multi_season', 'val_year': 2024, 'test_year': 2025, 'n_train': 119344, 'n_val': 119289, 'n_test': 120243}
Train: 119344, Val: 119289, Test: 120243


In [4]:
import xgboost as xgb
from sklearn.metrics import log_loss

clf = xgb.XGBClassifier(n_estimators=200, max_depth=6, random_state=42, eval_metric="mlogloss")
clf.fit(X_train[feature_cols], y_train, sample_weight=sample_weight_train)
y_pred = clf.predict(X_val[feature_cols])
macro_f1 = f1_score(y_val, y_pred, average="macro", zero_division=0)
print(f"Val macro F1: {macro_f1:.4f}")
print(classification_report(y_val, y_pred, target_names=FINAL_OUTCOME_NAMES, zero_division=0))
y_proba_test = clf.predict_proba(X_test[feature_cols])
print(f"Temporal test log-loss: {log_loss(y_test, y_proba_test, labels=list(range(len(FINAL_OUTCOME_NAMES)))):.4f}")
print(f"Temporal test macro F1: {f1_score(y_test, clf.predict(X_test[feature_cols]), average='macro', zero_division=0):.4f}")

Val macro F1: 0.6222
                precision    recall  f1-score   support

   in_play_out       0.89      0.76      0.82     79608
    in_play_1b       0.54      0.73      0.62     25782
extra_base_hit       0.32      0.40      0.36      8455
    in_play_hr       0.68      0.70      0.69      5444

      accuracy                           0.73    119289
     macro avg       0.61      0.65      0.62    119289
  weighted avg       0.76      0.73      0.74    119289

Temporal test log-loss: 0.6386
Temporal test macro F1: 0.6173


In [5]:
# Save model and metadata for app/simulator. Inference maps extra_base_hit -> in_play_2b for game state.
import json
import joblib

_SAVED = Path(_PATH) / "saved_models"
_SAVED.mkdir(parents=True, exist_ok=True)
_MODELS = _PROJECT_ROOT / "Models" / "saved_models"
_MODELS.mkdir(parents=True, exist_ok=True)

joblib.dump(clf, _SAVED / "final_outcome_model.joblib")
metadata = {
    "feature_cols": feature_cols,
    "class_names": FINAL_OUTCOME_NAMES,
    "hr_rule_distance_ft": 380,
    "hr_rule_launch_angle_min": 20,
    "hr_rule_launch_angle_max": 40,
}
with open(_SAVED / "final_outcome_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
if _MODELS.exists():
    import shutil
    shutil.copy(_SAVED / "final_outcome_model.joblib", _MODELS / "final_outcome_model.joblib")
    shutil.copy(_SAVED / "final_outcome_metadata.json", _MODELS / "final_outcome_metadata.json")
print("Saved final_outcome_model.joblib and final_outcome_metadata.json (4-class: out, single, extra_base_hit, hr)")

Saved final_outcome_model.joblib and final_outcome_metadata.json (4-class: out, single, extra_base_hit, hr)


### Optional: Two-stage hierarchical model
If the single 4-class model still underperforms, uncomment and run the cells below to train:
- **Stage 1:** Out vs Hit .
- **Stage 2:** On hits only, Single vs XBH vs HR.
This separates "contact quality" from "hit type".

In [6]:
#Two-stage: Stage 1 = Out (0) vs Hit (1); Stage 2 = Single (0) vs XBH (1) vs HR (2)
#y_train/y_val are 0=out, 1=1b, 2=xbh, 3=hr
y_bin_train = (y_train > 0).astype(int)
y_bin_val = (y_val > 0).astype(int)
hit_train = y_train > 0
hit_val = y_val > 0
# Among hits: 0=single, 1=xbh, 2=hr
y_hit_train = y_train[hit_train] - 1  # 1->0, 2->1, 3->2
y_hit_val = y_val[hit_val] - 1
X_hit_train = X_train[feature_cols].loc[hit_train]
X_hit_val = X_val[feature_cols].loc[hit_val]
# Stage 1: upweight Out by 20%; max_depth=8 so gatekeeper is "picky" (deep fly out vs HR ball)
weight_stage1 = np.array([1.2 if yi == 0 else 1.0 for yi in y_bin_train])
stage1 = xgb.XGBClassifier(n_estimators=150, max_depth=8, random_state=42, eval_metric="logloss")
stage1.fit(X_train[feature_cols], y_bin_train, sample_weight=weight_stage1)
# Stage 2: "Golden ratio" weights — single 1.2, XBH 1.8, HR 2.2 (compress so HR doesn't drown others)
STAGE2_WEIGHTS = [1.2, 1.8, 2.2]  # single, extra_base_hit, hr (golden ratio: compress so HR doesn't drown others)
stage2_weight = np.array([STAGE2_WEIGHTS[yi] for yi in y_hit_train])
stage2 = xgb.XGBClassifier(n_estimators=150, max_depth=5, random_state=43, eval_metric="mlogloss")
stage2.fit(X_hit_train, y_hit_train, sample_weight=stage2_weight)
# Validation: stage1 pred, then stage2 on predicted hits
bin_pred = stage1.predict(X_val[feature_cols])
hit_pred_mask = bin_pred == 1
y_val_4 = np.zeros_like(y_val)
y_val_4[~hit_pred_mask] = 0
if hit_pred_mask.any():
    y_val_4[hit_pred_mask] = 1 + stage2.predict(X_val[feature_cols].loc[hit_pred_mask])
print("Two-stage Val macro F1:", f1_score(y_val, y_val_4, average="macro", zero_division=0))
print(classification_report(y_val, y_val_4, target_names=FINAL_OUTCOME_NAMES, zero_division=0))
# To use two-stage in app: save stage1 + stage2 and set metadata "two_stage": True (see utils._resolve_hit_to_in_play)

Two-stage Val macro F1: 0.6025657312943671
                precision    recall  f1-score   support

   in_play_out       0.82      0.89      0.85     79608
    in_play_1b       0.63      0.55      0.59     25782
extra_base_hit       0.39      0.22      0.28      8455
    in_play_hr       0.68      0.71      0.69      5444

      accuracy                           0.76    119289
     macro avg       0.63      0.59      0.60    119289
  weighted avg       0.74      0.76      0.75    119289



In [7]:
# Save two-stage models and metadata so the app uses Out vs Hit -> Single/XBH/HR
import json
import joblib
import shutil

_SAVED = Path(_PATH) / "saved_models"
_MODELS = _PROJECT_ROOT / "Models" / "saved_models"
_SAVED.mkdir(parents=True, exist_ok=True)
_MODELS.mkdir(parents=True, exist_ok=True)

joblib.dump(stage1, _SAVED / "final_outcome_stage1_model.joblib")
joblib.dump(stage2, _SAVED / "final_outcome_stage2_model.joblib")
metadata = {
    "two_stage": True,
    "feature_cols": feature_cols,
    "class_names": FINAL_OUTCOME_NAMES,
    "hr_rule_distance_ft": 380,
    "hr_rule_launch_angle_min": 20,
    "hr_rule_launch_angle_max": 40,
    "hr_suppress_multiplier": 0.8,
}
with open(_SAVED / "final_outcome_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
for name in ["final_outcome_stage1_model.joblib", "final_outcome_stage2_model.joblib", "final_outcome_metadata.json"]:
    shutil.copy(_SAVED / name, _MODELS / name)
print("Saved two-stage models (stage1 + stage2) and metadata with two_stage=True.")

Saved two-stage models (stage1 + stage2) and metadata with two_stage=True.
